# 01 - Load Data + EDA

Loads the full Moltbook dataset (cached to `data/raw/posts.parquet`) and shows the CADFEB
content-category distribution split into two toxicity slices:

- **toxicity == 0** (safe) — population for the assistant judge in notebook 02.
- **toxicity in {3, 4}** (manipulative/malicious) — population for the persona judge in notebook 02.

Read-only against the cached raw dataset. Safe to re-run at any time.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, dataset

sns.set_theme(style="whitegrid")

In [ ]:
df = dataset.download_posts()
print(f"loaded {len(df)} rows, {df[config.COL_CATEGORY].nunique()} categories")
df[config.COL_TOXICITY].value_counts().sort_index()

## CADFEB category distribution, split by toxicity slice

In [ ]:
safe = df[(df[config.COL_TOXICITY] == config.TOXICITY_SAFE) & (df[config.COL_CATEGORY].isin(config.CADFEB_CATEGORIES))]
malicious = df[
    (df[config.COL_TOXICITY].isin(config.CADFEB_PERSONA_TOXICITY_LEVELS))
    & (df[config.COL_CATEGORY].isin(config.CADFEB_CATEGORIES))
]

safe_counts = safe[config.COL_CATEGORY].value_counts().reindex(config.CADFEB_CATEGORIES, fill_value=0)
malicious_counts = malicious[config.COL_CATEGORY].value_counts().reindex(config.CADFEB_CATEGORIES, fill_value=0)

print(f"toxicity==0 (safe): {len(safe)} posts across {config.CADFEB_CATEGORIES}")
print(safe_counts)
print()
print(f"toxicity in {config.CADFEB_PERSONA_TOXICITY_LEVELS} (manipulative/malicious): {len(malicious)} posts across {config.CADFEB_CATEGORIES}")
print(malicious_counts)

## Two views, side by side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

sns.barplot(x=safe_counts.index, y=safe_counts.values, ax=axes[0], color="#4C72B0")
axes[0].set_title("Category distribution -- toxicity == 0 (safe)")
axes[0].set_xlabel("content category")
axes[0].set_ylabel("post count")

sns.barplot(x=malicious_counts.index, y=malicious_counts.values, ax=axes[1], color="#C44E52")
axes[1].set_title("Category distribution -- toxicity in {3, 4} (manipulative/malicious)")
axes[1].set_xlabel("content category")
axes[1].set_ylabel("post count")

plt.tight_layout()
plt.show()

## Full toxicity x category breakdown

All toxicity levels and categories in the raw dataset, for context.

In [ ]:
df.groupby([config.COL_CATEGORY, config.COL_TOXICITY]).size().unstack(fill_value=0)